In [ ]:
# ============================================================
# FILE IPYNB DA ESEGUIRE SU COLAB PER LA GENERAZIONE DELLE RICOSTRUZIONI, (DATASET TV_RICOSTRUZIONI GIA' PORTATE NELLA REPOSITORY )
# 1. MOUNT DRIVE E ESTRAZIONE LOCALE SU SSD
# ============================================================
from google.colab import drive
import os
import shutil
from pathlib import Path

drive.mount('/content/drive')

# Modificate il percorso se lo zip su Drive è in una sottocartella
ZIP_DRIVE_PATH = "/content/drive/MyDrive/sinograms.zip"
LOCAL_SINOGRAMS_DIR = Path("/content/sinograms")
LOCAL_OUTPUT_DIR = Path("/content/tv_reconstructions")
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/tv_reconstructions")

if not LOCAL_SINOGRAMS_DIR.exists():
    print("Estraggo i sinogrammi nell'SSD locale...")
    get_ipython().system('unzip -q {ZIP_DRIVE_PATH} -d /content/')
else:
    print("Sinogrammi già estratti in locale, salto l'unzip.")


Mounted at /content/drive
Estraggo i sinogrammi nell'SSD locale...


In [3]:
# ============================================================
# 2. INSTALLAZIONE PACCHETTI (nessun import di IPPy qui dentro)
# ============================================================
get_ipython().system('pip install -q git+https://github.com/devangelista2/IPPy.git')
try:
    get_ipython().system('pip install -q cupy-cuda12x')
    import cupy  # noqa: F401
    print("CuPy installato correttamente.")
except Exception as e:
    print(f"CuPy non disponibile ({e}).")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.4/14.4 MB 28.7 MB/s eta 0:00:00
CuPy installato correttamente.


In [4]:
# ============================================================
# 2bis. PATCH DEL BUG DI IPPy — va eseguita PRIMA di "from IPPy import operators"
# ============================================================
import re, glob

candidates = glob.glob("/usr/**/dist-packages/IPPy/operators.py", recursive=True) + \
             glob.glob("/usr/**/site-packages/IPPy/operators.py", recursive=True)
assert candidates, "Non trovo operators.py di IPPy installato."
operators_path = candidates[0]

with open(operators_path) as f:
    src = f.read()

pattern = re.compile(
    r'(elif torch\.cuda\.is_available\(\) and not force_cpu:\s*\n'
    r'\s*warnings\.warn\(\s*\n'
    r'\s*"CUDA available but CuPy not found\. CTProjector limited to CPU operations for ASTRA data transfer\."\s*\n'
    r'\s*\)\s*\n'
    r'\s*# Force CPU mode if CuPy isn\'t there for GPU data handling\s*\n'
    r'(\s*)self\.use_gpu = False)'
)

def _fix(m):
    indent = m.group(2)
    return (
        "elif torch.cuda.is_available() and not force_cpu:\n"
        f"{indent}try:\n"
        f"{indent}    import cupy  # noqa: F401\n"
        f"{indent}except ImportError:\n"
        f"{indent}    warnings.warn(\n"
        f'{indent}        "CUDA available but CuPy not found. CTProjector limited to CPU operations for ASTRA data transfer."\n'
        f"{indent}    )\n"
        f"{indent}    self.use_gpu = False\n"
    )

new_src, n = pattern.subn(_fix, src)
if n == 1:
    with open(operators_path, "w") as f:
        f.write(new_src)
    print(f"Patch applicata a {operators_path}")
elif "import cupy  # noqa: F401" in src:
    print("Patch già presente, salto.")
else:
    raise RuntimeError(f"Blocco da patchare non trovato in {operators_path}.")

Patch applicata a /usr/local/lib/python3.13/dist-packages/IPPy/operators.py


In [5]:
# ============================================================
# 2ter. ORA sì, import di IPPy (dopo la patch, non prima)
# ============================================================
import torch
import numpy as np
from tqdm import tqdm
from IPPy import operators
from IPPy.solvers import ChambollePockTpVConstrained

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo in uso: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Dispositivo in uso: cuda
GPU: Tesla T4


In [6]:

# ============================================================
# 3. PARAMETRI E PROIETTORI SU GPU
# ============================================================
IMG_SIZE = (256, 256)
NOISE_LEVEL = 0.005
MAX_ITER = 300

ANGLE_CONFIGS = {
    90: np.linspace(-45, 45, 90),
    45: np.linspace(-45, 45, 45),
    30: np.linspace(-30, 30, 32)[1:-1],
    15: np.linspace(-30, 30, 17)[1:-1],
}

BEST_LAMBDAS = {
    90: 0.05,
    45: 0.05,
    30: 0.03,
    15: 0.03
}

PROJECTORS = {
    n_angles: operators.CTProjector(
        img_shape=IMG_SIZE,
        angles=np.deg2rad(angles),
        geometry="parallel",
        force_cpu=False,
    )
    for n_angles, angles in ANGLE_CONFIGS.items()
}

Attempting to create ASTRA projector type: 'cuda' for 'parallel' geometry...
Successfully created ASTRA projector type: 'cuda'
CTProjector initialized. Geometry: parallel. Using GPU: True. FBP Algorithm: FBP_CUDA
Attempting to create ASTRA projector type: 'cuda' for 'parallel' geometry...
Successfully created ASTRA projector type: 'cuda'
CTProjector initialized. Geometry: parallel. Using GPU: True. FBP Algorithm: FBP_CUDA
Attempting to create ASTRA projector type: 'cuda' for 'parallel' geometry...
Successfully created ASTRA projector type: 'cuda'
CTProjector initialized. Geometry: parallel. Using GPU: True. FBP Algorithm: FBP_CUDA
Attempting to create ASTRA projector type: 'cuda' for 'parallel' geometry...
Successfully created ASTRA projector type: 'cuda'
CTProjector initialized. Geometry: parallel. Using GPU: True. FBP Algorithm: FBP_CUDA


In [7]:
def quick_test():
    import time

    test_split = "train"
    test_n_angles = 90
    test_dir = LOCAL_SINOGRAMS_DIR / test_split / str(test_n_angles)
    test_paths = sorted(test_dir.rglob("*.npy"))
    if not test_paths:
        print("Nessun file trovato per il test rapido, salto.")
        return

    solver = ChambollePockTpVConstrained(PROJECTORS[test_n_angles])
    lmbda = BEST_LAMBDAS[test_n_angles]

    sinogram = np.load(test_paths[0])
    y_delta = torch.from_numpy(sinogram).float().unsqueeze(0).unsqueeze(0)
    epsilon = NOISE_LEVEL * torch.norm(y_delta)
    starting_point = torch.zeros((1, 1, *IMG_SIZE))

    t0 = time.time()
    x_sol, info = solver(
        y_delta,
        epsilon=epsilon,
        lmbda=lmbda,
        x_true=None,
        starting_point=starting_point,
        maxiter=MAX_ITER,
        p=1,
        verbose=False,
    )
    elapsed = time.time() - t0
    print(f"Tempo per una immagine ({test_n_angles} angoli, {MAX_ITER} iter): {elapsed:.2f} s")
    print(f"Iterazioni effettive: {info['iterations']} / {MAX_ITER}")   # <-- qui, dentro
    print(f"Stima per l'intero split '{test_split}' a {test_n_angles} angoli "
          f"({len(test_paths)} immagini): {elapsed * len(test_paths) / 3600:.1f} ore")

quick_test()

/usr/local/lib/python3.13/dist-packages/IPPy/solvers.py:127: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  nu = math.sqrt(


Tempo per una immagine (90 angoli, 300 iter): 5.69 s
Iterazioni effettive: 300 / 300
Stima per l'intero split 'train' a 90 angoli (2733 immagini): 4.3 ore


In [8]:

# ============================================================
# 5. UTILITY DI SINCRONIZZAZIONE INCREMENTALE SU DRIVE
# ============================================================
def sync_to_drive(src_dir: Path, dst_dir: Path):
    """Copia su Drive solo i file non ancora presenti (skip di quelli già sincronizzati)."""
    if not src_dir.exists():
        return
    dst_dir.mkdir(parents=True, exist_ok=True)
    n_copied = 0
    for f in src_dir.rglob("*.npy"):
        rel = f.relative_to(src_dir)
        dst = dst_dir / rel
        if dst.exists():
            continue
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, dst)
        n_copied += 1
    print(f"Sincronizzati {n_copied} nuovi file su Drive ({dst_dir}).")



In [9]:
# ============================================================
# 6. LOOP DI RICOSTRUZIONE TV SU TUTTI GLI SPLIT
# ============================================================
SYNC_EVERY = 100   # immagini tra una sincronizzazione su Drive e l'altra

splits = ["train", "validation", "test"]
failures = []

for split in splits:
    print(f"\n=================== Elaborazione Split: {split.upper()} ===================")

    for n_angles, K_config in PROJECTORS.items():
        print(f"\n--> Angoli: {n_angles} | Lambda: {BEST_LAMBDAS[n_angles]}")

        solver = ChambollePockTpVConstrained(K_config)
        lmbda = BEST_LAMBDAS[n_angles]

        input_sino_dir = LOCAL_SINOGRAMS_DIR / split / str(n_angles)
        local_out_dir = LOCAL_OUTPUT_DIR / split / str(n_angles)
        drive_out_dir = DRIVE_OUTPUT_DIR / split / str(n_angles)

        sino_paths = sorted(
            input_sino_dir.rglob("*.npy"),
            key=lambda p: int(p.stem)   # ordina per il numero vero, non per il testo
        )

        n_since_sync = 0
        new_files_batch = []

        for sino_path in tqdm(sino_paths, desc=f"TV Rec [{split}-{n_angles} deg]"):
            rel_path = sino_path.relative_to(input_sino_dir)
            local_path = (local_out_dir / rel_path).with_suffix(".npy")
            drive_path = (drive_out_dir / rel_path).with_suffix(".npy")

            if drive_path.exists():   # già su Drive, salta
                continue

            local_path.parent.mkdir(parents=True, exist_ok=True)

            try:
                sinogram = np.load(sino_path)
                y_delta = torch.from_numpy(sinogram).float().unsqueeze(0).unsqueeze(0)
                epsilon = NOISE_LEVEL * torch.norm(y_delta)
                starting_point = torch.zeros((1, 1, *IMG_SIZE))

                with torch.no_grad():
                    x_sol, _ = solver(
                        y_delta,
                        epsilon=epsilon,
                        lmbda=lmbda,
                        x_true=None,
                        starting_point=starting_point,
                        maxiter=MAX_ITER,
                        p=1,
                        verbose=False,
                    )

                x_sol_np = x_sol.squeeze().detach().numpy().astype(np.float32)
                np.save(local_path, x_sol_np)

            except Exception as e:
                failures.append((str(sino_path), str(e)))
                continue

            new_files_batch.append((local_path, rel_path))
            n_since_sync += 1

            if n_since_sync >= SYNC_EVERY:
                for lp, rp in new_files_batch:
                    dp = drive_out_dir / rp
                    dp.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(lp, dp)
                new_files_batch = []
                n_since_sync = 0

        # sync finale di quello che resta a fine blocco (l'ultimo pezzetto < SYNC_EVERY)
        for lp, rp in new_files_batch:
            dp = drive_out_dir / rp
            dp.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(lp, dp)

print("\nRicostruzione TV completata su tutti gli split!")

if failures:
    print(f"\n{len(failures)} immagini fallite:")
    for path, err in failures[:20]:
        print(f"  - {path}: {err}")
    if len(failures) > 20:
        print(f"  ... e altre {len(failures) - 20}")


=================== Elaborazione Split: TRAIN ===================

--> Angoli: 90 | Lambda: 0.05


TV Rec [train-90 deg]: 100%|██████████| 2733/2733 [00:11<00:00, 232.37it/s] 



--> Angoli: 45 | Lambda: 0.05


TV Rec [train-45 deg]: 100%|██████████| 2733/2733 [00:11<00:00, 248.12it/s] 



--> Angoli: 30 | Lambda: 0.03


...


--> Angoli: 30 | Lambda: 0.03


TV Rec [test-30 deg]: 100%|██████████| 327/327 [12:21<00:00,  2.27s/it]



--> Angoli: 15 | Lambda: 0.03


TV Rec [test-15 deg]: 100%|██████████| 327/327 [11:52<00:00,  2.18s/it]



Ricostruzione TV completata su tutti gli split!


In [10]:
print(f"Fallimenti finora: {len(failures)}")
for path, err in failures[:5]:
    print(path)
    print(" ->", err)
    print()

Fallimenti finora: 0
